# Agent 1: Query Analyzer

## Τι κάνει αυτός ο agent;

Ο **Query Analyzer** είναι το πρώτο node στον γράφο σας. Δέχεται την ερώτηση του χρήστη και αποφαίνεται:

| Τύπος ερώτησης | Παράδειγμα | Ενέργεια |
|---|---|---|
| **simple** | "What is attention mechanism?" | Περνάει as-is στον Retriever |
| **complex** | "Compare Transformers vs RNNs in terms of parallelism" | Διασπάται σε 2+ sub-queries |
| **out_of_scope** | "Who won the Champions League?" | Αρνείται, επιστρέφει μήνυμα |

## Η  ιδέα: ένα LLM, πολλοί agents

Όλοι οι agents χρησιμοποιούν **το ίδιο Ollama μοντέλο**.
Το διαφορετικό `system prompt` είναι αυτό που τους κάνει να συμπεριφέρονται διαφορετικά.

```
Ollama (llama3.2:3b)
    │
    ├── system prompt A ── Query Analyzer
    ├── system prompt B ── Retriever & Evaluator  
    └── system prompt C ── Synthesizer
```

## Structured output μέσω prompt engineering

Δεν χρησιμοποιούμε function calling. Αντί αυτού, ζητάμε από το μοντέλο να επιστρέφει **αυστηρά JSON** και το parse-άρουμε με Python.

Αυτό λειτουργεί αξιόπιστα αν:
1. Το system prompt είναι σαφές
2. Δείχνουμε παραδείγματα (few-shot)
3. Έχουμε fallback αν το JSON είναι invalid


In [7]:
# Βεβαιωθείτε ότι το Ollama τρέχει: ollama serve
# Και ότι έχετε κατεβάσει ένα μοντέλο: ollama pull llama3.2:3b

from openai import OpenAI
import json

OLLAMA_BASE  = "http://localhost:11434/v1"
OLLAMA_MODEL = "llama3.2:3b"  # αλλάξτε αν χρησιμοποιείτε άλλο μοντέλο

llm = OpenAI(base_url=OLLAMA_BASE, api_key="ollama")

def call_llm(system_prompt: str, user_message: str, temperature: float = 0.1) -> str:
    """Βοηθητική συνάρτηση: στέλνει μήνυμα στο Ollama και επιστρέφει την απάντηση."""
    response = llm.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content

# Quick sanity check
print(call_llm("You are a helpful assistant.", "Say 'OK' if you are working."))


OK


## Το System Prompt του Query Analyzer

Το prompt καθορίζει:
1. Τι ρόλο παίζει το μοντέλο
2. Ακριβώς ποιο format να επιστρέψει
3. Παραδείγματα για κάθε περίπτωση (few-shot examples)

> **Σημαντικό**: Το `temperature=0.1` (χαμηλό) κάνει το μοντέλο πιο προβλέψιμο
> στο format της απόκρισης. Για structured output, πάντα χρησιμοποιείτε χαμηλό temperature.


In [8]:
QUERY_ANALYZER_PROMPT = """
You are a Query Analyzer for an academic literature assistant.
The database contains papers about Artificial Intelligence and Machine Learning.

Given a user question, classify it and respond with ONLY valid JSON — no other text.

Rules:
- "simple": one focused question answerable with a single search
- "complex": requires searching multiple distinct topics separately
- "out_of_scope": not related to AI/ML academic papers

For "simple" questions, return:
{"type": "simple", "queries": ["<original or slightly rephrased question>"]}

For "complex" questions, decompose and return:
{"type": "complex", "queries": ["<subquery 1>", "<subquery 2>", ...]}

For "out_of_scope" questions, return:
{"type": "out_of_scope", "queries": [], "reason": "<brief explanation in Greek>"}

Examples:

Q: "How does the attention mechanism work in Transformers?"
A: {"type": "simple", "queries": ["attention mechanism in Transformers"]}

Q: "Compare the parallelization capabilities of Transformers and RNNs"
A: {"type": "complex", "queries": ["Transformer parallelization training", "RNN sequential computation limitations"]}

Q: "Who won the 2024 Champions League?"
A: {"type": "out_of_scope", "queries": [], "reason": "Η ερώτηση αφορά αθλητικά αποτελέσματα, όχι ακαδημαϊκά papers AI/ML."}
"""


## Η συνάρτηση `analyze_query`

Τυπώνει τη raw απόκριση, κάνει parse και έχει fallback αν το JSON είναι λάθος.


In [9]:
def extract_json(text: str) -> dict:
    """Εξάγει το πρώτο JSON object από κείμενο (χρήσιμο αν το μοντέλο προσθέτει εξτρά κείμενο)."""
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError(f"No JSON found in: {text!r}")
    return json.loads(text[start:end])

def analyze_query(question: str) -> dict:
    """
    Στέλνει την ερώτηση στο LLM και επιστρέφει:
    {
        "type":    "simple" | "complex" | "out_of_scope",
        "queries": [str, ...],       # 1+ queries
        "reason":  str | None        # μόνο για out_of_scope
    }
    """
    raw = call_llm(QUERY_ANALYZER_PROMPT, question)
    print(f"[QueryAnalyzer] raw response:\n{raw}\n")

    try:
        result = extract_json(raw)
        # Βεβαιωνόμαστε ότι υπάρχουν τα απαραίτητα πεδία
        assert "type" in result and "queries" in result
        return result
    except (ValueError, AssertionError, json.JSONDecodeError) as e:
        print(f"[QueryAnalyzer] JSON parse error: {e} — falling back to simple")
        return {"type": "simple", "queries": [question]}


## Δοκιμή με τις 3 κατηγορίες ερωτήσεων


In [10]:
test_questions = [
    # Simple
    "How does a spiking neural network differ from a traditional neural network?",
    # Complex
    "Compare the training efficiency and accuracy of Transformers versus RNNs for NLP tasks",
    # Out of scope
    "What is the capital of France?",
]

for q in test_questions:
    print("=" * 70)
    print(f"QUESTION: {q}")
    result = analyze_query(q)
    print(f"TYPE:     {result['type']}")
    print(f"QUERIES:  {result['queries']}")
    if result.get("reason"):
        print(f"REASON:   {result['reason']}")
    print()


QUESTION: How does a spiking neural network differ from a traditional neural network?
[QueryAnalyzer] raw response:
{"type": "complex", "queries": ["spiking neural network architecture", "traditional neural network differences"]}

TYPE:     complex
QUERIES:  ['spiking neural network architecture', 'traditional neural network differences']

QUESTION: Compare the training efficiency and accuracy of Transformers versus RNNs for NLP tasks
[QueryAnalyzer] raw response:
{"type": "complex", "queries": ["Transformer training efficiency NLP", "RNN training accuracy NLP", "Transformers vs RNNs in NLP", "NLP model parallelization"]}

TYPE:     complex
QUERIES:  ['Transformer training efficiency NLP', 'RNN training accuracy NLP', 'Transformers vs RNNs in NLP', 'NLP model parallelization']

QUESTION: What is the capital of France?
[QueryAnalyzer] raw response:
{"type": "simple", "queries": ["capital of France"]}

TYPE:     simple
QUERIES:  ['capital of France']



## Το node για LangGraph

Τώρα τυλίγουμε τη λογική σε LangGraph node function.
Η συνάρτηση διαβάζει το `question` από το state και γράφει τα αποτελέσματα.

> Σε ένα LangGraph node, **πάντα επιστρέφουμε dict** με τα πεδία που άλλαξαν.


In [11]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    question:    str
    query_type:  str        # "simple" | "complex" | "out_of_scope"
    queries:     list[str]  # 1+ queries που θα σταλούν στον Retriever
    answer:      str        # τελική απάντηση (γεμίζεται από τον Synthesizer)

def query_analyzer_node(state: AgentState) -> dict:
    """LangGraph node: αναλύει την ερώτηση και γεμίζει query_type + queries."""
    result = analyze_query(state["question"])

    if result["type"] == "out_of_scope":
        reason = result.get("reason", "Η ερώτηση είναι εκτός πεδίου.")
        return {
            "query_type": "out_of_scope",
            "queries":    [],
            "answer":     f"Εκτός πεδίου: {reason}",
        }

    return {
        "query_type": result["type"],
        "queries":    result["queries"],
    }

def route_after_analysis(state: AgentState) -> str:
    """Routing function: αν out_of_scope  END, αλλιώς  επόμενο agent."""
    if state["query_type"] == "out_of_scope":
        return "end"
    return "next_agent"

# Minimal graph μόνο με τον Query Analyzer (για δοκιμή)
def placeholder_next(state):
    """Placeholder για τον επόμενο agent (θα αντικατασταθεί στο notebook 05)."""
    return {"answer": f"[Placeholder] Θα επεξεργαστώ: {state['queries']}"}

g = StateGraph(AgentState)
g.add_node("query_analyzer",   query_analyzer_node)
g.add_node("placeholder_next", placeholder_next)

g.set_entry_point("query_analyzer")
g.add_conditional_edges(
    "query_analyzer",
    route_after_analysis,
    {"end": END, "next_agent": "placeholder_next"}
)
g.add_edge("placeholder_next", END)

app = g.compile()

INIT = {"question": "", "query_type": "", "queries": [], "answer": ""}

for q in [
    "What datasets are used to evaluate RAG systems?",
    "Compare spiking neural networks and LSTMs for speech recognition tasks",
    "What is the population of Tokyo?",
]:
    print("\n" + "=" * 70)
    print(f"Q: {q}")
    result = app.invoke({**INIT, "question": q})
    print(f"type:    {result['query_type']}")
    print(f"queries: {result['queries']}")
    print(f"answer:  {result['answer']}")



Q: What datasets are used to evaluate RAG systems?
[QueryAnalyzer] raw response:
{"type": "simple", "queries": ["RAG system evaluation datasets"]}

type:    simple
queries: ['RAG system evaluation datasets']
answer:  [Placeholder] Θα επεξεργαστώ: ['RAG system evaluation datasets']

Q: Compare spiking neural networks and LSTMs for speech recognition tasks
[QueryAnalyzer] raw response:
{"type": "complex", "queries": ["spiking neural network architecture for speech recognition", "LSTM architecture for speech recognition limitations", "speech recognition with spiking neural networks vs LSTMs"]}

type:    complex
queries: ['spiking neural network architecture for speech recognition', 'LSTM architecture for speech recognition limitations', 'speech recognition with spiking neural networks vs LSTMs']
answer:  [Placeholder] Θα επεξεργαστώ: ['spiking neural network architecture for speech recognition', 'LSTM architecture for speech recognition limitations', 'speech recognition with spiking neur

## Τι μάθαμε σε αυτό το notebook

1. **Ένα LLM + διαφορετικό system prompt = διαφορετικός agent**
2. **Structured output** μέσω prompt + JSON parsing + fallback
3. **Node function** που διαβάζει state και επιστρέφει dict
4. **Routing function** που αποφασίζει την επόμενη κατεύθυνση
5. **Early exit** για out-of-scope (άμεσα στο `END`)

---

Στο επόμενο notebook (`03_retriever_evaluator_agent.ipynb`) φτιάχνουμε τον Retriever & Evaluator
με το **ReAct loop**: retrieve  evaluate  reformulate  retry.
